In [16]:
%pip install -r ../requirements.txt


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    median_absolute_error,
    r2_score
)

In [18]:
df = pd.read_csv("../data/main_data_cleaned_with_cost.csv")

print("Rows:", len(df))
print("Unique projects:", df["project_id"].nunique())

df.head()

Rows: 141
Unique projects: 64


,project_id,substation_id,substation_name,state_code,state_name,latitude,longitude,project_name,cost,length_mi,estimated_in_service_year,change_type
0,2,1,Gore,VA,Virginia,39.276203,-78.364642,Rebuild Gore - Stonewall 138 kV with 500 kV Ov...,158800000,15,2028.0,Rebuild
1,2,2,Stonewall,VA,Virginia,39.272976,-78.083085,Rebuild Gore - Stonewall 138 kV with 500 kV Ov...,158800000,15,2028.0,Rebuild
2,3,1,Elizabeth City,NC,North Carolina,36.314098,-76.226179,Rebuild Shawboro - Elizabeth City 230 kV Line,30600000,10.3,2025.0,Rebuild
3,3,2,Shawboro,NC,North Carolina,36.384856,-76.080060,Rebuild Shawboro - Elizabeth City 230 kV Line,30600000,10.3,2025.0,Rebuild
4,4,1,Hornertown,NC,North Carolina,36.445990,-77.659058,Rebuild Hornertown - Hathaway 230 kV,49100000,28.9,2026.0,Rebuild


In [19]:
project_df = (
    df.groupby("project_id")
    .agg(
        project_name=("project_name", "first"),
        state_code=("state_code", "first"),
        cost=("cost", "first"),
        length_mi=("length_mi", "first"),
        estimated_in_service_year=("estimated_in_service_year", "first"),
        change_type=("change_type", "first"),
        num_substations=("substation_id", "nunique")
    )
    .reset_index()
)

print("Projects:", len(project_df))

Projects: 64


In [20]:
def clean_cost(value):
    value = str(value).strip()

    if "-" in value:
        low, high = value.split("-")
        return (float(low) + float(high)) / 2

    if value.startswith(">"):
        return np.nan

    return float(value)


project_df["cost_numeric"] = (
    project_df["cost"].apply(clean_cost)
)

project_df["length_mi"] = pd.to_numeric(
    project_df["length_mi"],
    errors="coerce"
)

project_df["estimated_in_service_year"] = pd.to_numeric(
    project_df["estimated_in_service_year"],
    errors="coerce"
)

In [21]:
change_types = {
    "rebuild": "Rebuild",
    "reconductor": "Reconductor",
    "upgrade": "Upgrade",
    "new_circuit_greenfield": "New circuit, greenfield",
    "new_circuit_existing_row": "New circuit, existing ROW"
}

for column, label in change_types.items():
    project_df[column] = (
        project_df["change_type"]
        .fillna("")
        .str.contains(label, regex=False)
        .astype(int)
    )

In [22]:
def extract_max_voltage(name):
    voltages = re.findall(
        r"(\d+)\s*kV",
        str(name),
        flags=re.IGNORECASE
    )

    if not voltages:
        return np.nan

    return max(map(float, voltages))


project_df["max_voltage_kv"] = (
    project_df["project_name"].apply(extract_max_voltage)
)

In [23]:
project_df["length_mi"] = (
    project_df["length_mi"]
    .fillna(project_df["length_mi"].median())
)

project_df["estimated_in_service_year"] = (
    project_df["estimated_in_service_year"]
    .fillna(project_df["estimated_in_service_year"].median())
)

project_df["max_voltage_kv"] = (
    project_df["max_voltage_kv"]
    .fillna(project_df["max_voltage_kv"].median())
)

project_df["log_cost"] = np.log1p(
    project_df["cost_numeric"]
)

In [24]:
features = [
    "length_mi",
    "estimated_in_service_year",
    "num_substations",
    "max_voltage_kv",
    "rebuild",
    "reconductor",
    "upgrade",
    "new_circuit_greenfield",
    "new_circuit_existing_row"
]

model_df = project_df.dropna(
    subset=["cost_numeric"]
).copy()

X = model_df[features]
y = model_df["log_cost"]

print("Usable training projects:", len(model_df))

Usable training projects: 63


In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

log_predictions = model.predict(X_test)

predicted_costs = np.expm1(log_predictions)
actual_costs = np.expm1(y_test)

mae = mean_absolute_error(
    actual_costs,
    predicted_costs
)

median_ae = median_absolute_error(
    actual_costs,
    predicted_costs
)

r2 = r2_score(
    y_test,
    log_predictions
)

print(f"Mean Absolute Error: ${mae:,.0f}")
print(f"Median Absolute Error: ${median_ae:,.0f}")
print(f"R² (log cost): {r2:.3f}")

Mean Absolute Error: $1,787,285,344
Median Absolute Error: $164,425,128
R² (log cost): 0.652


In [26]:
cv_scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="r2"
)

ridge_model = Ridge(alpha=1.0)

ridge_scores = cross_val_score(
    ridge_model,
    X,
    y,
    cv=5,
    scoring="r2"
)

print(
    "Random Forest average R²:",
    cv_scores.mean()
)

print(
    "Ridge average R²:",
    ridge_scores.mean()
)

Random Forest average R²: 0.5356587763539583
Ridge average R²: 0.29007861796977574


In [27]:
final_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

final_model.fit(X, y)

print(
    "Final model trained on",
    len(X),
    "projects."
)

Final model trained on 63 projects.


In [28]:
target_df = pd.read_csv(
    "../data/main_data_cleaned.csv"
)

valid_target = target_df[
    target_df["project_id"].notna()
    & target_df["project_name"].notna()
].copy()

target_projects = (
    valid_target.groupby("project_id")
    .agg(
        project_name=("project_name", "first"),
        state_code=("state_code", "first"),
        length_mi=("length_mi", "first"),
        estimated_in_service_year=(
            "estimated_in_service_year",
            "first"
        ),
        change_type=("change_type", "first"),
        num_substations=("substation_id", "nunique")
    )
    .reset_index()
)

print(
    "Projects to estimate:",
    len(target_projects)
)

Projects to estimate: 571


In [29]:
# Record missing values BEFORE filling them

target_projects["length_was_missing"] = (
    target_projects["length_mi"].isna()
)

target_projects["year_was_missing"] = (
    target_projects["estimated_in_service_year"].isna()
)

target_projects["change_type_was_missing"] = (
    target_projects["change_type"].isna()
)


# Create change-type features

for column, label in change_types.items():
    target_projects[column] = (
        target_projects["change_type"]
        .fillna("")
        .str.contains(label, regex=False)
        .astype(int)
    )


# Extract voltage from project name

target_projects["max_voltage_kv"] = (
    target_projects["project_name"]
    .apply(extract_max_voltage)
)

target_projects["voltage_was_missing"] = (
    target_projects["max_voltage_kv"].isna()
)


# Fill missing values using TRAINING-data medians

target_projects["length_mi"] = (
    target_projects["length_mi"]
    .fillna(project_df["length_mi"].median())
)

target_projects["estimated_in_service_year"] = (
    target_projects["estimated_in_service_year"]
    .fillna(
        project_df["estimated_in_service_year"].median()
    )
)

target_projects["max_voltage_kv"] = (
    target_projects["max_voltage_kv"]
    .fillna(project_df["max_voltage_kv"].median())
)

print(target_projects[features].isna().sum())

length_mi                    0
estimated_in_service_year    0
num_substations              0
max_voltage_kv               0
rebuild                      0
reconductor                  0
upgrade                      0
new_circuit_greenfield       0
new_circuit_existing_row     0
dtype: int64


In [30]:
X_target = target_projects[features]

target_log_predictions = (
    final_model.predict(X_target)
)

target_projects["estimated_cost"] = (
    np.expm1(target_log_predictions).round(0)
)

ValueError: could not convert string to float: 'Rebuild: 14 | Reconductor: 4 | Rebuild and reconductor: 5'

In [ ]:
# Create an input-data quality label for each estimate

missing_flags = [
    "length_was_missing",
    "year_was_missing",
    "voltage_was_missing",
    "change_type_was_missing"
]

target_projects["missing_feature_count"] = (
    target_projects[missing_flags].sum(axis=1)
)

target_projects["estimate_quality"] = pd.cut(
    target_projects["missing_feature_count"],
    bins=[-1, 0, 2, 4],
    labels=["higher", "moderate", "lower"]
)

print(target_projects["estimate_quality"].value_counts())

estimate_quality
higher      300
moderate    271
lower         0
Name: count, dtype: int64


In [ ]:
cost_predictions = target_projects[
    [
        "project_id",
        "estimated_cost",
        "estimate_quality"
    ]
]

final_df = target_df.merge(
    cost_predictions,
    on="project_id",
    how="left"
)

final_df["cost"] = final_df["estimated_cost"]

final_df["cost_source"] = "estimated"

final_df.loc[
    final_df["cost"].isna(),
    "cost_source"
] = None

final_df = final_df.drop(columns=["estimated_cost"])

output_path = "../data/main_data_with_estimated_costs.csv"

final_df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(final_df))
print("Rows with estimated cost:", final_df["cost"].notna().sum())
print("Rows without estimated cost:", final_df["cost"].isna().sum())

Saved: ../data/main_data_with_estimated_costs.csv
Rows: 1174
Rows with estimated cost: 1148
Rows without estimated cost: 26


In [32]:
training_df = pd.read_csv(
    "../data/main_data_cleaned_with_cost.csv"
)

estimated_df = pd.read_csv(
    "../data/main_data_with_estimated_costs.csv"
)

In [33]:
print("Training columns:")
print(training_df.columns.tolist())

print("\nEstimated columns:")
print(estimated_df.columns.tolist())

print(
    "\nOverlapping project IDs:",
    len(
        set(training_df["project_id"]) &
        set(estimated_df["project_id"])
    )
)

Training columns:
['project_id', 'substation_id', 'substation_name', 'state_code', 'state_name', 'latitude', 'longitude', 'project_name', 'cost', 'length_mi', 'estimated_in_service_year', 'change_type']

Estimated columns:
['project_id', 'substation_id', 'substation_name', 'state_code', 'state_name', 'latitude', 'longitude', 'project_name', 'cost', 'length_mi', 'estimated_in_service_year', 'change_type', 'estimate_quality', 'cost_source']

Overlapping project IDs: 0


In [34]:
# Label the known-cost projects
training_df["estimate_quality"] = None
training_df["cost_source"] = "known"

# Make sure columns are in the same order
training_df = training_df[estimated_df.columns]

# Combine known-cost + estimated-cost projects
final_df = pd.concat(
    [training_df, estimated_df],
    ignore_index=True
)

print("Known-cost projects:", len(training_df))
print("Estimated-cost projects:", len(estimated_df))
print("Final total:", len(final_df))

print("\nCost source:")
print(final_df["cost_source"].value_counts(dropna=False))

print("\nMissing costs:", final_df["cost"].isna().sum())
print("Duplicate project IDs:", final_df["project_id"].duplicated().sum())

Known-cost projects: 141
Estimated-cost projects: 1174
Final total: 1315

Cost source:
cost_source
estimated    1148
known         141
NaN            26
Name: count, dtype: int64

Missing costs: 26
Duplicate project IDs: 679


In [35]:
print("Exact duplicate rows:", final_df.duplicated().sum())

Exact duplicate rows: 0


In [36]:
output_path = "../data/main_data_final.csv"

final_df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(final_df))

Saved: ../data/main_data_final.csv
Rows: 1315
